# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"Loaded dataset - Name: {metadata.name}")
print(f"Description: {metadata.description}")

## 2. Data Overview
Review available record sets, their fields, and their `@id`s.

In [ ]:
# Inspect available record sets and their structure

record_sets = metadata.record_sets

if not record_sets:
    print("No record sets found in the metadata.")
else:
    for rs in record_sets:
        print(f"RecordSet @id: {rs.id if hasattr(rs, 'id') else rs['@id']}")
        print(f"  Name: {getattr(rs, 'name', None) or rs.get('name', '')}")
        if hasattr(rs, 'fields') or isinstance(rs.get('fields', None), list):
            fields = getattr(rs, 'fields', []) or rs.get('fields', [])
            print(f"  Fields:")
            for field in fields:
                fid = getattr(field, 'id', None) or field.get('@id', None)
                fname = getattr(field, 'name', None) or field.get('name', '')
                dtype = getattr(field, 'data_type', None) or field.get('dataType', None)
                print(f"    - @id: {fid}, name: {fname}, dataType: {dtype}")
        print()

## 3. Data Extraction
Load data from the primary record set into a DataFrame for analysis. **All entity references use their `@id`.**

In [ ]:
# Dynamically load all record sets as DataFrames
dataframes = {}

rs_ids = []
# Build list of record set @id's
if hasattr(metadata, 'record_sets') and metadata.record_sets:
    for rs in metadata.record_sets:
        rsid = getattr(rs, 'id', None) or rs.get('@id', None)
        if rsid:
            rs_ids.append(rsid)
else:
    print('No record sets found.')

print(f"Record sets found: {rs_ids}")

# For this dataset, there may be a main record set. If only one, use that one.
main_record_set_id = rs_ids[0] if rs_ids else None

for record_set in rs_ids:
    try:
        records = list(dataset.records(record_set=record_set))
        df = pd.DataFrame(records)
        dataframes[record_set] = df
        print(f"Loaded DataFrame for record set {record_set} with shape {df.shape}")
    except Exception as e:
        print(f"Error loading records for {record_set}: {e}")

if main_record_set_id is not None and main_record_set_id in dataframes:
    print(f"Columns in main record set [{main_record_set_id}]:")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records, normalizing numeric fields, and grouping data. All fields are referenced by their `@id`.

In [ ]:
# EDA: Filtering, normalization, grouping by field @id

if main_record_set_id and main_record_set_id in dataframes:
    df = dataframes[main_record_set_id].copy()
    # Select numeric fields for demonstration
    numeric_field_ids = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    print(f"Numeric fields: {numeric_field_ids}")
    if numeric_field_ids:
        numeric_field = numeric_field_ids[0]
        threshold = df[numeric_field].mean() if df[numeric_field].mean() != 0 else 10

        # Filter records
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        display(filtered_df.head())

        # Normalize
        norm_col = f"{numeric_field}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, norm_col]].head())

        # Try grouping by the first non-numeric field (as an example)
        group_field = None
        for col in df.columns:
            if col not in numeric_field_ids:
                group_field = col
                break

        if group_field is not None and group_field in filtered_df:
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"Grouped data by {group_field} (mean of numeric fields):")
            display(grouped_df.head())
    else:
        print("No numeric fields detected for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Below is an example histogram for a numeric field.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and main_record_set_id in dataframes:
    df = dataframes[main_record_set_id]
    numeric_field_ids = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if numeric_field_ids:
        field = numeric_field_ids[0]
        plt.figure(figsize=(8,4))
        sns.histplot(df[field].dropna(), bins=30, kde=True)
        plt.title(f"Distribution of {field}")
        plt.xlabel(field)
        plt.show()
    else:
        print("No numeric columns found for visualization.")

## 6. Conclusion
In this notebook, we used `mlcroissant` to load and explore the FAIR<sup>2</sup> dataset: Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells. We loaded data using only the official Croissant schema, programmatically inspected record sets and fields by their `@id`, extracted and transformed records using pandas, and visualized numeric distributions as a demonstration. All columns and fields referenced use the correct `@id` standard for clarity and interoperability.

Further analysis can extend these techniques to biological findings and advanced statistical summaries specific to proteomics.